# Preparação dos dados — M2

Este notebook demonstra a preparação reproduzível para modelagem: deduplicação, contexto pré-decisão, separação de ação/target/auditoria, splits estratificados e preprocessing ajustado somente no treino. Toda a lógica reside em `src/`; o notebook apresenta evidências e não cria estado oculto.

In [ ]:
from pathlib import Path
import json
import sys

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy import sparse

# Localiza a raiz independentemente do diretório inicial do kernel.
current_path = Path.cwd().resolve()
project_root = current_path if (current_path / 'configs').exists() else current_path.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.prepare import build_processed_datasets, load_preparation_config
from src.features.build_features import (
    ACTION_COLUMN,
    AUDIT_ONLY_COLUMNS,
    CONTEXT_COLUMNS,
    IDENTIFIER_COLUMN,
    TARGET_COLUMN,
)

data_config_path = project_root / 'configs' / 'data.yaml'
sns.set_theme(style='whitegrid', palette='deep')

## 1. Execução do pipeline

In [ ]:
# Regenera todos os outputs do M2 a partir da camada interim validada.
preparation_metadata = build_processed_datasets(data_config_path)
preparation_config = load_preparation_config(data_config_path)

pd.DataFrame([
    {
        'split': split_name,
        'registros': summary['row_count'],
        'positivos': summary['target_distribution']['1'],
        'taxa_positiva': summary['positive_rate'],
    }
    for split_name, summary in preparation_metadata['split']['summaries'].items()
])

A divisão usa 70%/15%/15%, `random_seed=42` e estratificação por `resultado`. O teste permanece intocado para a avaliação final.

## 2. Deduplicação e rastreabilidade

In [ ]:
pd.Series({
    'registros_entrada': preparation_metadata['input']['row_count'],
    'duplicatas_removidas': preparation_metadata['deduplication']['business_rows_removed'],
    'registros_restantes': preparation_metadata['deduplication']['remaining_rows'],
    'regra': preparation_metadata['deduplication']['keep'],
})

As duplicatas são detectadas sem `event_id`. A primeira ocorrência na ordem da fonte é mantida, e os identificadores removidos ficam registrados nos metadados.

## 3. Contrato do conjunto de modelagem

In [ ]:
train_frame = pd.read_csv(
    preparation_config.train_path,
    sep=preparation_config.csv_delimiter,
    encoding=preparation_config.csv_encoding,
)

pd.Series({
    'identificador': IDENTIFIER_COLUMN,
    'quantidade_contextos': len(CONTEXT_COLUMNS),
    'acao': ACTION_COLUMN,
    'target': TARGET_COLUMN,
    'duration_presente': 'duracao_contato' in train_frame.columns,
    'campaign_bruto_presente': 'contatos_campanha_atual' in train_frame.columns,
})

In [ ]:
train_frame[[IDENTIFIER_COLUMN, *CONTEXT_COLUMNS, ACTION_COLUMN, TARGET_COLUMN]].head()

`dias_desde_ultimo_contato=999` foi convertido em ausência imputável e no indicador `nunca_contatado_anteriormente`. `contatos_campanha_atual` foi substituído por `tentativas_anteriores_campanha_atual`, que remove o contato corrente da informação disponível.

## 4. Separação da camada de auditoria

In [ ]:
audit_frame = pd.read_csv(
    preparation_config.audit_path,
    sep=preparation_config.csv_delimiter,
    encoding=preparation_config.csv_encoding,
)
audit_frame[[IDENTIFIER_COLUMN, *AUDIT_ONLY_COLUMNS, 'split']].head()

Os campos demográficos e financeiros permanecem em arquivo separado para análise de disparidade. Eles não chegam ao `ColumnTransformer` nem controlam a política do MVP.

## 5. Preprocessing ajustado exclusivamente no treino

In [ ]:
preprocessor_artifact = joblib.load(preparation_config.preprocessor_path)
feature_names = preprocessor_artifact['feature_names']

pd.Series({
    'split_do_fit': preprocessor_artifact['fit_split'],
    'colunas_de_contexto': len(preprocessor_artifact['context_columns']),
    'features_transformadas': len(feature_names),
    'duration_no_contexto': 'duracao_contato' in preprocessor_artifact['context_columns'],
    'acao_no_contexto': ACTION_COLUMN in preprocessor_artifact['context_columns'],
    'target_no_contexto': TARGET_COLUMN in preprocessor_artifact['context_columns'],
})

In [ ]:
pd.DataFrame({'feature': feature_names})

In [ ]:
matrix_paths = {
    'train': preparation_config.train_matrix_path,
    'validation': preparation_config.validation_matrix_path,
    'test': preparation_config.test_matrix_path,
}
matrix_summary = pd.DataFrame([
    {
        'split': name,
        'linhas': sparse.load_npz(path).shape[0],
        'features': sparse.load_npz(path).shape[1],
    }
    for name, path in matrix_paths.items()
])
matrix_summary

O encoder usa `handle_unknown='ignore'`; os campos numéricos usam mediana e padronização calculadas apenas no treino. Validação e teste recebem somente `transform`.

## 6. Paridade do target nos splits

In [ ]:
split_summary = pd.DataFrame([
    {
        'split': name,
        'taxa_positiva': summary['positive_rate'],
    }
    for name, summary in preparation_metadata['split']['summaries'].items()
])

figure, axis = plt.subplots(figsize=(7, 4))
sns.barplot(data=split_summary, x='split', y='taxa_positiva', ax=axis)
axis.set(title='Taxa positiva por split', xlabel='Split', ylabel='Taxa positiva')
axis.yaxis.set_major_formatter(lambda value, position: f'{value:.1%}')
plt.show()

## 7. Conclusão

- 12 duplicatas foram removidas com linhagem preservada.
- Os 41.176 eventos restantes foram divididos sem sobreposição.
- A taxa positiva permaneceu em aproximadamente 11,27% nos três splits.
- Ação, target, identificador, auditoria, duração e campanha bruta não entram no contexto.
- O preprocessing foi ajustado apenas no treino e gerou 27 features.
- Os hashes, versões, nomes de features, shapes e IDs removidos estão registrados em `preparation.metadata.json`.

O M3 poderá consumir esses splits para construir baseline determinístico, baseline preditivo e política adaptativa sem refazer decisões de preparação.